# Week 2 Sorting Complexity Lab (Teacher Companion)

This notebook mirrors the Week 2 learning sheet.

- Recreates the comparison / swap counts for each sorting algorithm.
- Provides helper traces that you can re-run with different sequences.
- Confirms why the tallied numbers (28, 120, 26, 110, etc.) appear in the teacher guide.

Feel free to tweak the sample arrays to match classroom data.


## Sample Data Sets

We use the same sequences referenced in the teacher playbook:

- `A8`, `A16`, `A32`: progressively longer integer lists for sorting demos.
- `G5`, `G8`, `G12`: adjacency matrices for Prim's algorithm cost counting (simple dense graphs).


## Big-O Cheat Sheet
- **Bubble Sort**: time `O(n^2)` (best `O(n)` if nearly sorted), space `O(1)` in-place.
- **Cocktail Sort**: similar to bubble → time `O(n^2)` (best `O(n)`), space `O(1)`.
- **Insertion Sort**: time `O(n^2)` worst/average, best `O(n)`; space `O(1)`.
- **Heap Sort**: time `O(n log n)` any case; space `O(1)` using array heap.
- **Prim's Algorithm** (array implementation): time `O(n^2)`, space `O(n)` for arrays (visited/parent). With priority queue → `O(E log V)` time, `O(V)` space.


In [1]:
from copy import deepcopy
from math import inf

A8 = [5, 2, 9, 1, 6, 3, 8, 4]
A16 = [5, 2, 9, 1, 6, 3, 8, 4, 11, 7, 15, 12, 14, 10, 13, 16]
A32 = [5, 2, 9, 1, 6, 3, 8, 4, 11, 7, 15, 12, 14, 10, 13, 16,
       21, 18, 24, 19, 22, 17, 20, 23, 30, 25, 29, 26, 28, 27, 31, 32]

G5 = [
    [0, 4, 2, 0, 0],
    [4, 0, 5,10, 0],
    [2, 5, 0, 3, 4],
    [0,10, 3, 0, 6],
    [0, 0, 4, 6, 0],
]

G8 = [
    [0, 4, 2, 0, 7, 0, 0, 0],
    [4, 0, 5,10, 8, 0, 0, 0],
    [2, 5, 0, 3, 4, 9, 0, 0],
    [0,10, 3, 0, 6, 0,11, 0],
    [7, 8, 4, 6, 0, 5, 0, 9],
    [0, 0, 9, 0, 5, 0, 4, 7],
    [0, 0, 0,11, 0, 4, 0, 2],
    [0, 0, 0, 0, 9, 7, 2, 0],
]

G12 = [
    [0, 4, 2, 0, 7, 0, 0, 0, 8, 0, 0, 5],
    [4, 0, 5,10, 8, 0, 0, 0, 6, 0, 9, 0],
    [2, 5, 0, 3, 4, 9, 0, 0, 7, 5, 0, 4],
    [0,10, 3, 0, 6, 0,11, 0, 0, 0, 6, 0],
    [7, 8, 4, 6, 0, 5, 0, 9, 0, 0, 4, 3],
    [0, 0, 9, 0, 5, 0, 4, 7, 0, 6, 0, 8],
    [0, 0, 0,11, 0, 4, 0, 2, 0, 5, 7, 0],
    [0, 0, 0, 0, 9, 7, 2, 0, 4, 0, 0, 6],
    [8, 6, 7, 0, 0, 0, 0, 4, 0, 5, 0, 0],
    [0, 0, 5, 0, 0, 6, 5, 0, 5, 0, 3, 4],
    [0, 9, 0, 6, 4, 0, 7, 0, 0, 3, 0, 2],
    [5, 0, 4, 0, 3, 8, 0, 6, 0, 4, 2, 0],
]

print(f"Lengths: A8={len(A8)}, A16={len(A16)}, A32={len(A32)}")


Lengths: A8=8, A16=16, A32=32


## Trace Helpers

Each function returns `(comparisons, swaps_or_moves, extra_info)` so you can inspect passes or parents.


In [2]:
def bubble_trace(arr):
    a = deepcopy(arr)
    compares = swaps = 0
    n = len(a)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            compares += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swaps += 1
    return compares, swaps, {}


def cocktail_trace(arr):
    a = deepcopy(arr)
    compares = swaps = 0
    n = len(a)
    start, end = 0, n - 1
    swapped = True
    passes = []
    while swapped:
        swapped = False
        right = left = 0
        for i in range(start, end):
            compares += 1
            right += 1
            if a[i] > a[i + 1]:
                a[i], a[i + 1] = a[i + 1], a[i]
                swaps += 1
                swapped = True
        if not swapped:
            passes.append((right, 0))
            break
        swapped = False
        end -= 1
        for i in range(end - 1, start - 1, -1):
            compares += 1
            left += 1
            if a[i] > a[i + 1]:
                a[i], a[i + 1] = a[i + 1], a[i]
                swaps += 1
                swapped = True
        start += 1
        passes.append((right, left))
    return compares, swaps, {"passes": passes}


def insertion_trace(arr):
    a = deepcopy(arr)
    compares = moves = 0
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:
            compares += 1
            a[j + 1] = a[j]
            moves += 1
            j -= 1
        if j >= 0:
            compares += 1
        a[j + 1] = key
    return compares, moves, {}


def heap_trace(arr):
    a = deepcopy(arr)
    compares = swaps = 0
    n = len(a)

    def sift_down(i, size):
        nonlocal compares, swaps
        while 2 * i + 1 < size:
            child = 2 * i + 1
            if child + 1 < size:
                compares += 1
                if a[child] < a[child + 1]:
                    child += 1
            compares += 1
            if a[i] >= a[child]:
                break
            a[i], a[child] = a[child], a[i]
            swaps += 1
            i = child

    for i in range(n // 2 - 1, -1, -1):
        sift_down(i, n)
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        swaps += 1
        sift_down(0, end)
    return compares, swaps, {}


def prim_trace(matrix):
    n = len(matrix)
    visited = [False] * n
    low = [inf] * n
    parent = [-1] * n
    compares = moves = 0
    low[0] = 0
    for _ in range(n):
        u = -1
        for v in range(n):
            compares += 1
            if not visited[v] and (u == -1 or low[v] < low[u]):
                u = v
        visited[u] = True
        moves += 1
        for v in range(n):
            w = matrix[u][v]
            if w and not visited[v] and w < low[v]:
                compares += 1
                low[v] = w
                parent[v] = u
    return compares, moves, {"parent": parent}


## Reproducing the Tables

Run the cells below to compare the notebook output with the numbers in the guide.


In [3]:
def summarize_sort(name, arrs, tracer):
    rows = []
    for label, seq in arrs.items():
        comp, ops, extra = tracer(seq)
        rows.append((label, comp, ops, extra))
    return name, rows

algorithms = [
    summarize_sort("Bubble", {"A8": A8, "A16": A16, "A32": A32}, bubble_trace),
    summarize_sort("Cocktail", {"A8": A8, "A16": A16, "A32": A32}, cocktail_trace),
    summarize_sort("Insertion", {"A8": A8, "A16": A16, "A32": A32}, insertion_trace),
    summarize_sort("Heap", {"A8": A8, "A16": A16, "A32": A32}, heap_trace),
]
for name, rows in algorithms:
    print(f"{name} Sort")
    for label, comp, ops, extra in rows:
        print(f"  {label}: comparisons={comp}, swaps/moves={ops}")
        if extra:
            print(f"    extra={extra}")


Bubble Sort
  A8: comparisons=28, swaps/moves=13
  A16: comparisons=120, swaps/moves=24
  A32: comparisons=496, swaps/moves=46
Cocktail Sort
  A8: comparisons=25, swaps/moves=13
    extra={'passes': [(7, 6), (5, 4), (3, 0)]}
  A16: comparisons=65, swaps/moves=24
    extra={'passes': [(15, 14), (13, 12), (11, 0)]}
  A32: comparisons=145, swaps/moves=46
    extra={'passes': [(31, 30), (29, 28), (27, 0)]}
Insertion Sort
  A8: comparisons=18, swaps/moves=13
  A16: comparisons=37, swaps/moves=24
  A32: comparisons=75, swaps/moves=46
Heap Sort
  A8: comparisons=25, swaps/moves=19
  A16: comparisons=76, swaps/moves=54
  A32: comparisons=225, swaps/moves=140


### Prim's Algorithm Checks


In [4]:
for label, graph in {"G5": G5, "G8": G8, "G12": G12}.items():
    comp, moves, extra = prim_trace(graph)
    print(f"{label}: comparisons={comp}, selected={moves}")


G5: comparisons=29, selected=5
G8: comparisons=76, selected=8
G12: comparisons=172, selected=12


## Things to Try

- Swap in a nearly sorted list (e.g., `sorted(A16)`), rerun `bubble_trace`, and notice the comparisons drop to `n-1`.
- Shuffle the values to see how swaps change while comparisons stay tied to the formula.
- Replace the dense Prim matrices with a sparse one to gauge how real problem structure affects the raw counts.
